# Unified Phase Retrieval Core

This notebook demonstrates `library/phase_retrieval_core_unified.py`: ordinary two-helicity use, arbitrary labeled hologram dictionaries, and the `Nmodes` switch between the fast single-mode kernel and the multimode kernel.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "library" / "phase_retrieval_core_unified.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find library/phase_retrieval_core_unified.py")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



import h5py
import pandas as pd
from ipydatagrid import DataGrid




from library import phase_retrieval_core_unified as pr
from library import interactive


import json
import h5py
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets


def recipe_to_df(recipe):
    max_len = max(len(v) if isinstance(v, list) else 1 for v in recipe.values())

    rows = {}
    for key, value in recipe.items():
        if isinstance(value, list):
            rows[key] = value + [None] * (max_len - len(value))
        else:
            rows[key] = [value] + [None] * (max_len - 1)

    return pd.DataFrame.from_dict(
        rows,
        orient="index",
        columns=[f"step_{i}" for i in range(max_len)]
    )


def parse_cell(x):
    if x == "":
        return None
    try:
        return json.loads(x)
    except Exception:
        return x


def df_to_recipe(df):
    recipe = {}

    for key, row in df.iterrows():
        values = [parse_cell(v) for v in row.tolist()]

        while len(values) > 1 and values[-1] is None:
            values.pop()

        recipe[key] = values[0] if len(values) == 1 else values

    return recipe


def save_recipe_hdf5(recipe, filename):
    df = recipe_to_df(recipe)

    with h5py.File(filename, "w") as f:
        f.attrs["columns"] = json.dumps(list(df.columns))
        f.attrs["index"] = json.dumps(list(df.index))

        dt = h5py.string_dtype(encoding="utf-8")
        data = df.map(lambda x: json.dumps(x)).to_numpy(dtype=object)

        f.create_dataset("table", data=data, dtype=dt)


def load_recipe_hdf5(filename):
    with h5py.File(filename, "r") as f:
        columns = json.loads(f.attrs["columns"])
        index = json.loads(f.attrs["index"])

        raw = f["table"][()]
        data = [
            [
                json.loads(c.decode("utf-8") if isinstance(c, bytes) else c)
                for c in row
            ]
            for row in raw
        ]

    df = pd.DataFrame(data, index=index, columns=columns)
    return df_to_recipe(df), df


def edit_recipe(recipe, filename="recipe.h5"):
    df = recipe_to_df(recipe)

    cells = {}
    rows = []

    header = [widgets.HTML("<b>key</b>")]
    header += [widgets.HTML(f"<b>{c}</b>") for c in df.columns]
    rows.append(widgets.HBox(header))

    for key in df.index:
        row_widgets = [widgets.HTML(f"<b>{key}</b>", layout=widgets.Layout(width="260px"))]

        for col in df.columns:
            value = df.loc[key, col]
            text = "" if value is None else json.dumps(value)

            w = widgets.Text(
                value=text,
                layout=widgets.Layout(width="110px")
            )
            cells[(key, col)] = w
            row_widgets.append(w)

        rows.append(widgets.HBox(row_widgets))

    out = widgets.Output()

    def collect_df():
        df_new = df.copy()
        for (key, col), widget in cells.items():
            df_new.loc[key, col] = widget.value
        return df_new

    def on_save(_):
        df_new = collect_df()
        recipe_new = df_to_recipe(df_new)
        save_recipe_hdf5(recipe_new, filename)

        with out:
            clear_output()
            print(f"Saved to {filename}")

    save_button = widgets.Button(description="Save recipe", button_style="success")
    save_button.on_click(on_save)

    display(widgets.VBox(rows + [save_button, out]))

    return cells

## Synthetic Inputs

Replace these arrays with centered measured hologram intensities. All holograms in the dictionary must have the same `(nx, ny)` shape.

In [ ]:
rng = np.random.default_rng(3)
shape = (64, 64)
yy, xx = np.indices(shape)
rr = np.hypot(xx - shape[1] / 2, yy - shape[0] / 2)




mask_pixel = np.zeros(shape, dtype=int)
#mask_pixel[30:34, 30:34] = 1
supportmask = (rr < 5).astype(float)
#supportmask = ((rr) < 5).astype(float)

base = 2.0 + xx-yy+rr+10*np.sin(yy/5)+np.exp(-(rr / 12) ** 2)
base*=supportmask
pos = np.abs(interactive.reconstruct(base))**2
base = 1.0 + rr*xx+4*np.sin(xx/10)+np.exp(-(rr-xx / 3) ** 2)
base*=supportmask
neg = np.abs(interactive.reconstruct(base))**2
base = 2.0j +rr+10*np.sin(yy*xx +xx/5)+ np.exp(-(rr-yy*xx / 18) ** 2)
base*=supportmask
LH = np.abs(interactive.reconstruct(base))**2
base = 1.0j +10*np.sin(yy*xx/5)+ np.exp(-(rr+xx-yy / 22) ** 2)
base*=supportmask
LV = np.abs(interactive.reconstruct(base))**2


holograms = {"pos": pos, "neg": neg, "LH": LH, "LV": LV}


## Labeled Hologram Sequence

The recipe key `helicity` is kept for compatibility with the old recipe format. In this module it means the hologram dictionary key used at each step.

In [ ]:
recipe = {
    "algorithm_list": ["HAPRE","ER", "ER", "ER", "ER"],
    "number_iterations": [700,50, 20, 20, 20],
    "helicity": ["pos","pos", "neg", "LH", "LV"],
    "beta_zero": [0.5,0.5, 0.5, 0.5, 0.5],
    "beta_mode": ["arctan","const", "const", "const", "const"],
    "alpha_zero": [0.0, 0.0, 0.0, 0.0, 0.0],
    "alpha_mode": ["const","const", "const", "const", "const"],
    "RL_its": [0,0, 0, 0, 0],
    "RL_freqs": [1e9,1e9, 1e9, 1e9, 1e9],
    "TV_freqs": [1e9,1e9, 1e9, 1e9, 1e9],
    "plot_every": [2,2, 2, 2, 2],
    "average_img": [2,2, 2, 2, 2],
    "Fourier_last": [True,True, True, True, True],
    "Startimage": [None, "pos","pos", "pos", "pos"],
    "Startgamma": [None, None, None, None, None],
    "Nmodes": 1,
    "normalize_startimage_between_holograms": True,
}



In [ ]:

cells = edit_recipe(recipe, "recipe.h5")

In [ ]:
cells

In [ ]:
recipe_edited = df_to_recipe(df_loaded)
save_recipe_hdf5(recipe_edited, "recipe.h5")

In [ ]:
save_recipe_hdf5(recipe, "recipe.h5")

recipe_loaded, df_loaded = load_recipe_hdf5("recipe.h5")


In [ ]:

result = pr.phase_retrieval_algorithm(holograms, mask_pixel, supportmask, recipe)
result.keys()

In [ ]:
fig, axes = plt.subplots(np.unique(recipe["helicity"]).size,2, figsize=(3, 6))
for ax, label in zip(axes, np.unique(recipe["helicity"])):
    field = result["full_coherence"][label]
    ax[0].imshow(np.log10(np.abs(field) + 1e-9), cmap="magma")
    ax[0].set_title(label)
    ax[0].axis("off")
    ax[1].imshow(np.abs(interactive.reconstruct(field)), cmap="magma")
    ax[1].set_title(label)
    ax[1].axis("off")
plt.tight_layout()

## Multimode Switch

`Nmodes == 1` uses the fast single-mode core. `Nmodes > 1` uses the multimode summed-intensity constraint.

In [ ]:
multi_recipe = recipe | {
    "Nmodes": 2,
    "Startimage": [
        np.stack([np.sqrt(pos), 1j * np.sqrt(pos)]) / np.sqrt(2),
        "pos",
        "neg",
        "LH",
    ],
}
multi_result = pr.phase_retrieval_algorithm(holograms, mask_pixel, supportmask, multi_recipe)
multi_result["recipe"]["Nmodes"], multi_result["full_coherence"]["LV"].shape